# AI_FILTER — Customer Feedback Filtering

`AI_FILTER` returns TRUE/FALSE based on a natural-language condition — a semantic WHERE clause.

| Item | Detail |
|---|---|
| **Function** | `SNOWFLAKE.CORTEX.AI_FILTER` |
| **Data** | 20 VARIANT JSON customer feedback records |
| **Use-case** | Semantic filtering without writing complex pattern-matching logic |


## Step 1 — Environment & Table Setup

In [ ]:
CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

In [ ]:
CREATE OR REPLACE TABLE customer_feedback (
    feedback_id   INT AUTOINCREMENT,
    product_name  VARCHAR(200),
    channel       VARCHAR(50),
    feedback      VARIANT,
    submitted_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

In [ ]:
INSERT INTO customer_feedback (product_name, channel, feedback) VALUES
('Mobile App', 'in-app', PARSE_JSON('{"text":"The new checkout flow is confusing and I lost my cart items twice","rating":2,"user_tier":"premium"}')),
('Mobile App', 'in-app', PARSE_JSON('{"text":"Love the new dark mode! Finally easy on the eyes at night","rating":5,"user_tier":"free"}')),
('Mobile App', 'email', PARSE_JSON('{"text":"App crashes every time I try to upload a photo","rating":1,"user_tier":"premium"}')),
('Web Dashboard', 'support', PARSE_JSON('{"text":"The export to CSV feature is broken since last update","rating":1,"user_tier":"enterprise"}')),
('Web Dashboard', 'survey', PARSE_JSON('{"text":"Great improvement in loading speed. Charts render much faster now","rating":4,"user_tier":"premium"}')),
('Web Dashboard', 'in-app', PARSE_JSON('{"text":"Would be nice to have more customizable widgets on the homepage","rating":3,"user_tier":"free"}')),
('API Platform', 'email', PARSE_JSON('{"text":"Rate limiting is too aggressive. Our batch jobs keep failing","rating":2,"user_tier":"enterprise"}')),
('API Platform', 'support', PARSE_JSON('{"text":"Documentation is excellent. Integrated in less than a day","rating":5,"user_tier":"premium"}')),
('API Platform', 'in-app', PARSE_JSON('{"text":"Getting timeout errors on the /search endpoint during peak hours","rating":1,"user_tier":"enterprise"}')),
('Mobile App', 'survey', PARSE_JSON('{"text":"Push notifications are too frequent and not relevant to me","rating":2,"user_tier":"free"}')),
('Mobile App', 'support', PARSE_JSON('{"text":"Biometric login works great. Very smooth experience","rating":5,"user_tier":"premium"}')),
('Web Dashboard', 'email', PARSE_JSON('{"text":"SSO integration broke after your maintenance window last week","rating":1,"user_tier":"enterprise"}')),
('Web Dashboard', 'survey', PARSE_JSON('{"text":"I appreciate the new color-coded alerts. Very intuitive","rating":4,"user_tier":"premium"}')),
('API Platform', 'support', PARSE_JSON('{"text":"Webhooks are unreliable. We miss about 5 percent of events","rating":2,"user_tier":"enterprise"}')),
('API Platform', 'email', PARSE_JSON('{"text":"The new GraphQL endpoint is a game changer for our frontend team","rating":5,"user_tier":"premium"}')),
('Mobile App', 'in-app', PARSE_JSON('{"text":"Battery drain is insane when the app runs in background","rating":1,"user_tier":"free"}')),
('Web Dashboard', 'support', PARSE_JSON('{"text":"Role-based access controls are exactly what we needed","rating":5,"user_tier":"enterprise"}')),
('Mobile App', 'survey', PARSE_JSON('{"text":"Decent app overall but nothing special compared to competitors","rating":3,"user_tier":"free"}')),
('API Platform', 'in-app', PARSE_JSON('{"text":"SDK for Python is well-designed. Clean API surface","rating":4,"user_tier":"premium"}')),
('Web Dashboard', 'email', PARSE_JSON('{"text":"Scheduled reports never arrive on time. Defeats the purpose","rating":2,"user_tier":"enterprise"}'));

In [ ]:
SELECT feedback_id, product_name, channel, feedback:text::STRING AS text, feedback:rating::INT AS rating
FROM customer_feedback LIMIT 5;

## Step 2 — Semantic Filtering

Use `AI_FILTER` with natural-language conditions instead of regex or LIKE patterns.

In [ ]:
SELECT
    feedback_id,
    product_name,
    feedback:text::STRING AS feedback_text
FROM customer_feedback
WHERE SNOWFLAKE.CORTEX.AI_FILTER(
    feedback:text::STRING,
    'describes a software bug or technical malfunction'
);

In [ ]:
SELECT
    feedback_id,
    product_name,
    feedback:text::STRING AS feedback_text
FROM customer_feedback
WHERE SNOWFLAKE.CORTEX.AI_FILTER(
    feedback:text::STRING,
    'contains a feature request or product suggestion'
);

In [ ]:
SELECT
    feedback_id,
    product_name,
    feedback:user_tier::STRING AS tier,
    feedback:text::STRING AS feedback_text
FROM customer_feedback
WHERE feedback:user_tier::STRING = 'enterprise'
  AND SNOWFLAKE.CORTEX.AI_FILTER(
    feedback:text::STRING,
    'expresses urgency, frustration, or reports a service outage'
  );

## Key Takeaways

- `AI_FILTER` acts as a semantic WHERE clause — no regex or keyword lists needed.
- Combine with traditional SQL filters (tier, product) for precise targeting.
- Perfect for routing tickets, flagging escalations, and triaging feedback.
- Describe the condition in plain English — the LLM handles interpretation.
